# 03 - Model Experiments and Validation

**Dynamic Pricing Engine** — predictive-model validation and decision-support design

This notebook documents the production modeling choices:

1. Train the leakage-safe XGBoost calendar-unavailability model
2. Evaluate a strict future-date holdout
3. Inspect feature importance and probability-quality metrics
4. Explain why observational price elasticity was retired
5. Demonstrate the assumption-based pricing scenario engine
6. Fit the anomaly detector
7. Inspect TreeSHAP explanations
8. Save only production models

> **Target semantics:** the source column is still named `was_booked` for backward
> compatibility, but it is treated as a **calendar-unavailability proxy**, not proof
> of a confirmed booking.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

from src.models.anomaly_detector import AnomalyDetector
from src.models.demand_forecaster import DEMAND_FEATURES, DemandForecaster
from src.models.scenario_engine import (
    SENSITIVITY_PRESETS,
    compute_scenario_curve,
    evaluate_sensitivity_presets,
)
from src.models.train_models import _even_parquet_sample
from src.utils.config import PROJECT_ROOT, load_config

config = load_config()
processed_dir = PROJECT_ROOT / config["data"]["processed_dir"]

## 1. Load a reproducible modeling sample

In [ ]:
MAX_ROWS = 500_000

calendar_path = processed_dir / "calendar_features.parquet"
listings_path = processed_dir / "listings_features.parquet"

calendar_columns = list(
    dict.fromkeys(["listing_id", "date", "was_booked"] + DEMAND_FEATURES)
)

calendar = _even_parquet_sample(
    calendar_path,
    calendar_columns,
    MAX_ROWS,
)

calendar["date"] = pd.to_datetime(calendar["date"], errors="coerce")
calendar = (
    calendar.dropna(subset=["date"])
    .sort_values(["date", "listing_id"], kind="mergesort")
    .reset_index(drop=True)
)

listing_candidates = [
    "id",
    "room_type_encoded",
    "beds",
    "bathrooms",
    "amenity_score",
    "review_score",
    "location_cluster",
]
listing_schema = set(pq.ParquetFile(listings_path).schema.names)
listing_columns = [c for c in listing_candidates if c in listing_schema]
listings = pd.read_parquet(listings_path, columns=listing_columns)

print(f"Calendar rows: {len(calendar):,}")
print(f"Listings represented: {calendar['listing_id'].nunique():,}")
print(f"Date range: {calendar['date'].min()} -> {calendar['date'].max()}")
print(f"Unique dates: {calendar['date'].nunique():,}")

## 2. Leakage-safe demand forecasting

The production model excludes target-history features such as
`rolling_7d_occupancy`, `rolling_30d_occupancy`, and `occupancy_rate`.

Validation is chronological: complete dates are kept together, and the final
future block is held out from model fitting.

In [ ]:
forecaster = DemandForecaster(config=config)

metrics = forecaster.train(
    calendar,
    listings,
    tune_hyperparams=False,
)

print("Demand Forecaster Metrics")
for key, value in metrics.items():
    if isinstance(value, (int, float)):
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value}")

print("\nTemporal split metadata")
for key, value in forecaster.split_metadata.items():
    print(f"  {key}: {value}")

### Metrics to emphasize

For a probability model, ROC-AUC alone is not enough. The production evaluation
also records:

- **PR-AUC** — ranking quality under class imbalance
- **Log loss** — penalizes confident wrong probabilities
- **Brier score** — squared error of predicted probabilities
- **Future holdout positive rate** — context for interpreting those scores

Use the untouched future holdout metrics as the headline benchmark rather than
the cross-validation mean.

In [ ]:
importance = forecaster.get_feature_importance()

print("Top 15 features")
print(importance.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
importance.head(15).plot.barh(
    x="feature",
    y="importance",
    ax=ax,
    legend=False,
)
ax.set_title("XGBoost Feature Importance — Calendar Unavailability Model")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Why the original elasticity estimator was retired

The original project fit a single observational log-log Ridge regression and
used its price coefficient as if it were a stable causal elasticity.

That interpretation was rejected during validation because price in marketplace
data is endogenous: hosts change price in response to expected demand, season,
location, listing quality, events, and other factors. A predictive association
between price and calendar availability is therefore **not automatically a
causal price-response estimate**.

The production system now separates:

- **Prediction:** XGBoost estimates a calendar-unavailability probability from
  observed listing/context features.
- **Decision support:** pricing scenarios use explicit, user-visible sensitivity
  assumptions.
- **Causal claims:** none are made from this observational dataset.

## 4. Assumption-based pricing scenarios

In [ ]:
base_demand_proxy = 0.65
reference_price = 150.0

scenario_results = evaluate_sensitivity_presets(
    base_demand_proxy=base_demand_proxy,
    reference_price=reference_price,
    config=config,
)

rows = []
for name, result in scenario_results.items():
    rows.append(
        {
            "scenario": name,
            "assumed_sensitivity": result.sensitivity,
            "reference_price": result.reference_price,
            "recommended_price": result.recommended_price,
            "demand_proxy_at_recommended": result.demand_proxy_at_recommended,
            "revenue_proxy_at_reference": result.revenue_proxy_at_reference,
            "revenue_proxy_at_recommended": result.revenue_proxy_at_recommended,
            "revenue_proxy_change_pct": result.revenue_proxy_change_pct,
        }
    )

scenario_table = pd.DataFrame(rows)
scenario_table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for name, sensitivity in SENSITIVITY_PRESETS.items():
    curve = compute_scenario_curve(
        base_demand_proxy=base_demand_proxy,
        reference_price=reference_price,
        sensitivity=sensitivity,
        config=config,
    )
    ax.plot(
        curve["prices"],
        curve["revenue_proxies"],
        label=f"{name} sensitivity ({sensitivity:.2f})",
    )

ax.axvline(reference_price, linestyle="--", label="reference price")
ax.set_title("Assumption-Based Revenue-Proxy Scenarios")
ax.set_xlabel("Candidate price")
ax.set_ylabel("Revenue proxy = price × demand proxy")
ax.legend()
plt.tight_layout()
plt.show()

The curves above are **scenario outputs**, not forecasts of realized revenue.
Their purpose is to show how recommendations change when the assumed price
sensitivity changes.

## 5. Anomaly detection

In [ ]:
detector = AnomalyDetector(contamination=0.05, config=config)
detector.fit(calendar)

anomalies = detector.predict(calendar)
print(
    f"Total anomalies: {anomalies.sum():,} / {len(anomalies):,} "
    f"({anomalies.mean() * 100:.1f}%)"
)

## 6. TreeSHAP explainability

In [ ]:
try:
    import shap

    feature_names = forecaster.feature_names
    model_frame = forecaster._prepare_features(calendar, listings)[0]
    X_sample = model_frame[feature_names].fillna(0).sample(
        min(500, len(model_frame)),
        random_state=42,
    )

    explainer = shap.TreeExplainer(forecaster.model)
    shap_values = explainer(X_sample)

    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title("TreeSHAP — Calendar Unavailability Model")
    plt.tight_layout()
    plt.show()

    shap.plots.waterfall(shap_values[0])

except ImportError:
    print("SHAP is not installed.")
except Exception as exc:
    print(f"SHAP analysis error: {exc}")

## 7. Save production models

In [ ]:
forecaster.save()
detector.save()
print("Saved demand forecaster and anomaly detector.")

## 8. Production model summary

| Component | Method | Role | Validation |
|---|---|---|---|
| Demand forecaster | XGBoost classifier | Predict calendar-unavailability probability | Whole-date CV + future holdout |
| Scenario engine | Explicit exponential sensitivity assumptions | Decision support across low/moderate/high price response | Synthetic known-response tests |
| Anomaly detector | Isolation Forest | Flag unusual observations | Contamination-based validation |
| Explainability | TreeSHAP | Explain XGBoost predictions | Tree-model additive contributions |

### Design decisions

- Target-history leakage features are excluded from the demand model.
- The final benchmark comes from a strictly later temporal holdout.
- Probability quality is evaluated with ROC-AUC, PR-AUC, log loss, and Brier score.
- Observational price association is not presented as causal elasticity.
- Pricing recommendations are conditional on explicit sensitivity assumptions.
- Only production components are persisted for serving.